# LeetCode 1546 — Maximum Number of Non-Overlapping Subarrays With Sum Equals Target bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-non-overlapping-subarrays-with-sum-equals-target/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxNonOverlapping(self, nums: List[int], target: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Dùng **tổng tiền tố** để biết nhanh tổng của một đoạn con: `sum(nums[j:i]) = pre[i] − pre[j]`.
2. Viết baseline **quy hoạch động `O(n²)`**: `dp[i]` = số đoạn tối đa trong `nums[:i]`.
3. Tối ưu DP thành `O(n)` bằng **hash map** "tiền tố → chỉ số gần nhất".
4. Nắm lời giải **tham lam**: quét trái → phải, gặp đoạn có tổng `target` **kết thúc sớm nhất** thì lấy ngay và **reset** tập tiền tố — và chứng minh được vì sao tham lam tối ưu (bài toán chọn khoảng).
5. Tránh các bẫy: đếm cả đoạn chồng lấn, quên `0` trong tập tiền tố, dùng cửa sổ trượt khi có số âm.

## 2. Đề bài & ví dụ

Cho mảng `nums` và số nguyên `target`. Trả về **số lượng lớn nhất** các đoạn con (liên tiếp) **khác rỗng**, **không chồng lấn nhau**, sao cho tổng mỗi đoạn bằng `target`.

**Ví dụ 1:** `nums = [1,1,1,1,1]`, `target = 2` → `2` (ví dụ `[1,1]` ở vị trí 0–1 và 2–3).

**Ví dụ 2:** `nums = [-1,3,5,1,4,2,-9]`, `target = 6` → `2`. Có 3 đoạn tổng 6: `[5,1]`, `[4,2]`, `[3,5,1,4,2,-9]`, nhưng chỉ hai đoạn đầu không chồng lấn.

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`, `−10⁴ ≤ nums[i] ≤ 10⁴`, `0 ≤ target ≤ 10⁶`.

In [1]:
import random
import time
from itertools import accumulate
from typing import List

examples = [([1, 1, 1, 1, 1], 2, 2), ([-1, 3, 5, 1, 4, 2, -9], 6, 2)]
for nums, target, want in examples:
    print(f"nums = {nums}, target = {target} -> mong đợi {want}")

nums = [1, 1, 1, 1, 1], target = 2 -> mong đợi 2
nums = [-1, 3, 5, 1, 4, 2, -9], target = 6 -> mong đợi 2


Hai ví dụ thường gặp khác của bài (kiểm chứng bằng DP brute force ở mục 12; nên đối chiếu lại đề gốc): `[-2,6,6,3,5,4,1,2,8]`, `target = 10` → `3` và `[0,0,0]`, `target = 0` → `3`.

In [2]:
more = [([-2, 6, 6, 3, 5, 4, 1, 2, 8], 10, 3), ([0, 0, 0], 0, 3)]
print(more)

[([-2, 6, 6, 3, 5, 4, 1, 2, 8], 10, 3), ([0, 0, 0], 0, 3)]


## 3. Kiến thức nền

### 3.1 Tổng tiền tố

`pre[0] = 0`, `pre[i] = nums[0] + … + nums[i−1]`. Tổng đoạn `nums[j..i−1]` = `pre[i] − pre[j]`. `itertools.accumulate` tính nhanh.

In [3]:
nums = [-1, 3, 5, 1, 4, 2, -9]
pre = [0] + list(accumulate(nums))
print("pre =", pre)
print("tổng nums[2..3] =", pre[4] - pre[2], "| kiểm tra:", sum(nums[2:4]))

pre = [0, -1, 2, 7, 8, 12, 14, 5]
tổng nums[2..3] = 6 | kiểm tra: 6


### 3.2 Đoạn kết thúc tại `i` có tổng `target` ⇔ tồn tại tiền tố bằng `pre[i] − target`

`pre[i] − pre[j] = target` ⇔ `pre[j] = pre[i] − target`. Vậy với mỗi vị trí kết thúc, chỉ cần tra xem giá trị `pre[i] − target` đã xuất hiện trong các tiền tố trước đó chưa — tra `set`/`dict` `O(1)`.

In [4]:
target = 6
seen = {0}
for i, p in enumerate(pre[1:], start=1):
    hit = (p - target) in seen
    print(f"i={i}: pre={p:>3}, cần {p - target:>3} -> {'CÓ đoạn tổng 6 kết thúc ở ' + str(i - 1) if hit else 'không'}")
    seen.add(p)

i=1: pre= -1, cần  -7 -> không
i=2: pre=  2, cần  -4 -> không
i=3: pre=  7, cần   1 -> không
i=4: pre=  8, cần   2 -> CÓ đoạn tổng 6 kết thúc ở 3
i=5: pre= 12, cần   6 -> không
i=6: pre= 14, cần   8 -> CÓ đoạn tổng 6 kết thúc ở 5
i=7: pre=  5, cần  -1 -> CÓ đoạn tổng 6 kết thúc ở 6


### 3.3 `dict` lưu chỉ số **gần nhất** của mỗi tiền tố

`last[value] = i` ghi đè mỗi lần gặp lại — luôn giữ vị trí mới nhất. Dùng trong DP `O(n)`.

In [5]:
last = {}
for i, p in enumerate(pre):
    last[p] = i
print(last)

{0: 0, -1: 1, 2: 2, 7: 3, 8: 4, 12: 5, 14: 6, 5: 7}


### 3.4 Bài toán chọn khoảng (interval scheduling)

Cho nhiều khoảng `[l, r]`, chọn nhiều khoảng nhất không giao nhau: sắp theo **đầu mút phải** tăng dần, lấy khoảng đầu tiên rồi bỏ những khoảng giao với nó, lặp lại. Đây là nền tảng chứng minh tham lam của bài.

In [6]:
intervals = [(1, 4), (2, 3), (3, 5), (6, 7), (5, 8)]
chosen, end = [], -1
for l, r in sorted(intervals, key=lambda x: x[1]):
    if l > end:
        chosen.append((l, r))
        end = r
print("chọn:", chosen)

chọn: [(2, 3), (6, 7)]


### 3.5 Vì sao **không** dùng cửa sổ trượt (two pointers)?

Cửa sổ trượt cần tổng **tăng đơn điệu** khi mở rộng (mọi số dương). Ở đây có số âm và số 0, nên thu hẹp/mở rộng cửa sổ không còn đúng (lỗi 4 ở mục 13).

In [7]:
print("Có số âm trong ví dụ 2:", any(x < 0 for x in examples[1][0]))

Có số âm trong ví dụ 2: True


## 4. Ý tưởng nền tảng / trực giác

Liệt kê tay **mọi** đoạn có tổng `6` trong ví dụ 2 dưới dạng khoảng chỉ số:

- `[2, 3]` = `5 + 1`
- `[4, 5]` = `4 + 2`
- `[1, 6]` = `3 + 5 + 1 + 4 + 2 − 9`

Chọn nhiều khoảng nhất không giao nhau: `[2,3]` và `[4,5]` ⇒ 2. Nhận xét: khoảng **kết thúc sớm nhất** (`[2,3]`) luôn nên được chọn — nó "chiếm chỗ" ít nhất về phía phải, để lại nhiều chỗ nhất cho các khoảng sau. Sau khi chọn, mọi khoảng bắt đầu ≤ 3 đều bị loại — tương đương **bắt đầu lại** việc tính tiền tố từ vị trí 4.

In [8]:
def all_target_intervals(nums, target):
    n = len(nums)
    pre = [0] + list(accumulate(nums))
    return [(j, i - 1) for i in range(1, n + 1) for j in range(i) if pre[i] - pre[j] == target]

print(all_target_intervals(*examples[1][:2]))
print(all_target_intervals(*examples[0][:2]))

[(2, 3), (4, 5), (1, 6)]
[(0, 1), (1, 2), (2, 3), (3, 4)]


## 5. Cách 1 — Brute Force / baseline: DP `O(n²)`

`dp[i]` = số đoạn tối đa (không chồng lấn, tổng `target`) nằm trong `nums[:i]`.

- Không dùng phần tử `i − 1` làm điểm kết thúc đoạn: `dp[i] = dp[i − 1]`.
- Có đoạn `nums[j..i−1]` tổng `target`: `dp[i] = max(dp[i], dp[j] + 1)`.

Duyệt mọi `j < i` bằng tổng tiền tố ⇒ `O(n²)`.

In [9]:
def brute_dp(nums, target):
    n = len(nums)
    pre = [0] + list(accumulate(nums))
    dp = [0] * (n + 1)
    for i in range(1, n + 1):
        dp[i] = dp[i - 1]
        for j in range(i):
            if pre[i] - pre[j] == target:
                dp[i] = max(dp[i], dp[j] + 1)
    return dp[n]

for nums, target, want in examples + more:
    print(nums, target, "->", brute_dp(nums, target), "mong đợi", want)

[1, 1, 1, 1, 1] 2 -> 2 mong đợi 2
[-1, 3, 5, 1, 4, 2, -9] 6 -> 2 mong đợi 2
[-2, 6, 6, 3, 5, 4, 1, 2, 8] 10 -> 3 mong đợi 3
[0, 0, 0] 0 -> 3 mong đợi 3


**Độ phức tạp:** Time `O(n²)` (`5·10⁹` phép với `n = 10⁵` — quá chậm), Space `O(n)`.

**Nút thắt:** với mỗi `i`, ta duyệt **mọi** `j`, trong khi chỉ những `j` có `pre[j] = pre[i] − target` mới có ích — và vì `dp` không giảm, trong số đó chỉ cần `j` **lớn nhất**.

In [10]:
rng0 = random.Random(0)
arr = [rng0.randint(-5, 5) for _ in range(2000)]
t0 = time.perf_counter()
r = brute_dp(arr, 3)
print(f"n=2000: {r} ({time.perf_counter() - t0:.2f}s) — n=10^5 sẽ chậm hơn ~2500 lần")

n=2000: 350 (0.08s) — n=10^5 sẽ chậm hơn ~2500 lần


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (DP):** `dp` không giảm theo `i` ⇒ trong các `j` thoả `pre[j] = pre[i] − target`, chọn `j` lớn nhất là tốt nhất ⇒ lưu `last[giá trị tiền tố] = chỉ số gần nhất`.

**Insight 2 (tham lam):** đây là bài chọn khoảng; luôn chọn đoạn có **đầu mút phải nhỏ nhất**. Quét trái → phải, đoạn đầu tiên được phát hiện (tại vị trí `i` nhỏ nhất có `pre[i] − target` thuộc tập tiền tố "hợp lệ") chính là đoạn kết thúc sớm nhất.

**Quy tắc tham lam:**
- Giữ `seen` = tập tiền tố **kể từ điểm cắt cuối cùng**, ban đầu `{0}`, và `cur` = tổng kể từ điểm cắt.
- Mỗi phần tử: `cur += x`. **Nếu** `cur − target ∈ seen` **thì** `ans += 1`, rồi reset `seen = {0}`, `cur = 0` (mọi đoạn sau phải bắt đầu sau điểm cắt). **Ngược lại** thêm `cur` vào `seen`.

**Invariant:** `seen` chứa đúng các tổng tiền tố (tính từ điểm cắt) của các vị trí bắt đầu còn hợp lệ; `ans` là số đoạn tối đa trong phần đã quét.

## 7. Các cách tiếp theo

### Cách 2 — DP + hash map, `O(n)`

`dp[i] = max(dp[i−1], dp[last[pre[i] − target]] + 1)` nếu khoá tồn tại. Cập nhật `last[pre[i]] = i` **sau** khi tính `dp[i]` (để `j < i`, tránh đoạn rỗng khi `target = 0`).

In [11]:
def dp_hash(nums, target):
    n = len(nums)
    dp = [0] * (n + 1)
    last = {0: 0}              # tiền tố 0 tại chỉ số 0
    p = 0
    for i in range(1, n + 1):
        p += nums[i - 1]
        dp[i] = dp[i - 1]
        j = last.get(p - target)
        if j is not None:
            dp[i] = max(dp[i], dp[j] + 1)
        last[p] = i            # cập nhật SAU khi dùng
    return dp[n]

for nums, target, want in examples + more:
    print(nums, target, "->", dp_hash(nums, target), "mong đợi", want)

[1, 1, 1, 1, 1] 2 -> 2 mong đợi 2
[-1, 3, 5, 1, 4, 2, -9] 6 -> 2 mong đợi 2
[-2, 6, 6, 3, 5, 4, 1, 2, 8] 10 -> 3 mong đợi 3
[0, 0, 0] 0 -> 3 mong đợi 3


In [12]:
class SolutionDP:
    def maxNonOverlapping(self, nums: List[int], target: int) -> int:
        n = len(nums)
        dp = [0] * (n + 1)
        last = {0: 0}
        p = 0
        for i, x in enumerate(nums, 1):
            p += x
            dp[i] = dp[i - 1]
            if p - target in last:
                dp[i] = max(dp[i], dp[last[p - target]] + 1)
            last[p] = i
        return dp[n]

print([SolutionDP().maxNonOverlapping(a, t) for a, t, _ in examples + more])

[2, 2, 3, 3]


**Độ phức tạp:** Time `O(n)` trung bình (tra `dict`), Space `O(n)`.

### Cách 3 — Tham lam: prefix sum + `set` reset khi tìm thấy

Đoạn code ngắn và là lời giải chuẩn:
- `seen = {0}`, `cur = 0`.
- Gặp đoạn ⇒ cắt ngay, reset cả `seen` và `cur`.

In [13]:
def greedy_reset(nums, target):
    ans, cur, seen = 0, 0, {0}
    for x in nums:
        cur += x
        if cur - target in seen:
            ans += 1
            cur, seen = 0, {0}       # bắt đầu lại sau điểm cắt
        else:
            seen.add(cur)
    return ans

for nums, target, want in examples + more:
    print(nums, target, "->", greedy_reset(nums, target), "mong đợi", want)

[1, 1, 1, 1, 1] 2 -> 2 mong đợi 2
[-1, 3, 5, 1, 4, 2, -9] 6 -> 2 mong đợi 2
[-2, 6, 6, 3, 5, 4, 1, 2, 8] 10 -> 3 mong đợi 3
[0, 0, 0] 0 -> 3 mong đợi 3


**Độ phức tạp:** Time `O(n)` trung bình, Space `O(n)` worst (khi không tìm thấy đoạn nào, `seen` lớn dần). Tổng số phần tử từng được thêm vào các `seen` là `≤ n`.

### Cách 4 — Tham lam không reset: `dict` chỉ số gần nhất + mốc cắt

Thay vì xoá `seen`, giữ `last[tiền tố toàn cục] = chỉ số gần nhất` và biến `boundary` = chỉ số tiền tố tại điểm cắt cuối. Một đoạn kết thúc tại `i` hợp lệ ⇔ `last[pre[i] − target] ≥ boundary` (bắt đầu sau điểm cắt). Tránh tạo `set` mới nhiều lần — cách nhìn gần với cách 2.

In [14]:
class SolutionBoundary:
    def maxNonOverlapping(self, nums: List[int], target: int) -> int:
        last = {0: 0}
        boundary = 0            # chỉ số tiền tố của điểm cắt gần nhất
        p, ans = 0, 0
        for i, x in enumerate(nums, 1):
            p += x
            j = last.get(p - target, -1)
            if j >= boundary:   # đoạn (j, i] nằm hoàn toàn sau điểm cắt
                ans += 1
                boundary = i
            last[p] = i
        return ans

print([SolutionBoundary().maxNonOverlapping(a, t) for a, t, _ in examples + more])

[2, 2, 3, 3]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 5 — Liệt kê khoảng + chọn khoảng theo đầu mút phải (minh hoạ chứng minh)

Liệt kê mọi đoạn tổng `target` (`O(n²)`), sắp theo đầu mút phải, chọn tham lam như mục 3.4. Chậm, nhưng thể hiện trực tiếp lý thuyết đứng sau cách 3.

In [15]:
def interval_greedy(nums, target):
    ivs = all_target_intervals(nums, target)
    ans, end = 0, -1
    for l, r in sorted(ivs, key=lambda x: x[1]):
        if l > end:
            ans += 1
            end = r
    return ans

for nums, target, want in examples + more:
    print(nums, target, "->", interval_greedy(nums, target), "mong đợi", want)

[1, 1, 1, 1, 1] 2 -> 2 mong đợi 2
[-1, 3, 5, 1, 4, 2, -9] 6 -> 2 mong đợi 2
[-2, 6, 6, 3, 5, 4, 1, 2, 8] 10 -> 3 mong đợi 3
[0, 0, 0] 0 -> 3 mong đợi 3


**Độ phức tạp:** Time `O(n² log n)` (có thể tới `O(n²)` khoảng), Space `O(n²)`.

## 8. TRACE từng bước

Trace cách 3 trên `[-2,6,6,3,5,4,1,2,8]`, `target = 10`:

| i | x | cur | cur − 10 | trong seen? | hành động | seen sau | ans |
|---|---|-----|----------|-------------|-----------|----------|-----|
| 0 | −2 | −2 | −12 | không | thêm | {0,−2} | 0 |
| 1 | 6 | 4 | −6 | không | thêm | {0,−2,4} | 0 |
| 2 | 6 | 10 | 0 | **có** | cắt (đoạn [0..2] = −2+6+6) | {0} | 1 |
| 3 | 3 | 3 | −7 | không | thêm | {0,3} | 1 |
| 4 | 5 | 8 | −2 | không | thêm | {0,3,8} | 1 |
| 5 | 4 | 12 | 2 | không | thêm | {0,3,8,12} | 1 |
| 6 | 1 | 13 | 3 | **có** | cắt (đoạn [4..6] = 5+4+1) | {0} | 2 |
| 7 | 2 | 2 | −8 | không | thêm | {0,2} | 2 |
| 8 | 8 | 10 | 0 | **có** | cắt (đoạn [7..8] = 2+8) | {0} | 3 |

Ở dòng i = 2, tiền tố cần tìm là `0` — giá trị ban đầu — nên đoạn là `[0..2] = −2 + 6 + 6 = 10`. Hàm `trace_greedy` in đúng đoạn được chọn (dùng thêm `dict` để nhớ vị trí bắt đầu).

In [16]:
def trace_greedy(nums, target):
    ans, cur = 0, 0
    pos = {0: -1}                    # tiền tố (từ điểm cắt) -> chỉ số kết thúc của tiền tố đó
    chosen = []
    print(f"{'i':>2} | {'x':>3} | {'cur':>4} | {'cur-t':>5} | hành động")
    for i, x in enumerate(nums):
        cur += x
        need = cur - target
        if need in pos:
            l = pos[need] + 1
            chosen.append((l, i))
            ans += 1
            print(f"{i:>2} | {x:>3} | {cur:>4} | {need:>5} | CẮT: đoạn [{l}..{i}] = {nums[l:i + 1]} tổng {sum(nums[l:i + 1])} | ans={ans}")
            cur, pos = 0, {0: i}
        else:
            pos[cur] = i
            print(f"{i:>2} | {x:>3} | {cur:>4} | {need:>5} | thêm {cur} vào seen = {sorted(pos)}")
    print("Các đoạn được chọn:", chosen)
    return ans

print("Kết quả:", trace_greedy([-2, 6, 6, 3, 5, 4, 1, 2, 8], 10))

 i |   x |  cur | cur-t | hành động
 0 |  -2 |   -2 |   -12 | thêm -2 vào seen = [-2, 0]
 1 |   6 |    4 |    -6 | thêm 4 vào seen = [-2, 0, 4]
 2 |   6 |   10 |     0 | CẮT: đoạn [0..2] = [-2, 6, 6] tổng 10 | ans=1
 3 |   3 |    3 |    -7 | thêm 3 vào seen = [0, 3]
 4 |   5 |    8 |    -2 | thêm 8 vào seen = [0, 3, 8]
 5 |   4 |   12 |     2 | thêm 12 vào seen = [0, 3, 8, 12]
 6 |   1 |   13 |     3 | CẮT: đoạn [4..6] = [5, 4, 1] tổng 10 | ans=2
 7 |   2 |    2 |    -8 | thêm 2 vào seen = [0, 2]
 8 |   8 |   10 |     0 | CẮT: đoạn [7..8] = [2, 8] tổng 10 | ans=3
Các đoạn được chọn: [(0, 2), (4, 6), (7, 8)]
Kết quả: 3


In [17]:
print("Kết quả:", trace_greedy([-1, 3, 5, 1, 4, 2, -9], 6))

 i |   x |  cur | cur-t | hành động
 0 |  -1 |   -1 |    -7 | thêm -1 vào seen = [-1, 0]
 1 |   3 |    2 |    -4 | thêm 2 vào seen = [-1, 0, 2]
 2 |   5 |    7 |     1 | thêm 7 vào seen = [-1, 0, 2, 7]
 3 |   1 |    8 |     2 | CẮT: đoạn [2..3] = [5, 1] tổng 6 | ans=1
 4 |   4 |    4 |    -2 | thêm 4 vào seen = [0, 4]
 5 |   2 |    6 |     0 | CẮT: đoạn [4..5] = [4, 2] tổng 6 | ans=2
 6 |  -9 |   -9 |   -15 | thêm -9 vào seen = [-9, 0]
Các đoạn được chọn: [(2, 3), (4, 5)]
Kết quả: 2


## 9. Code nộp LeetCode

Bản nộp chính — cách 3 (tham lam + `set` reset), đúng chữ ký `maxNonOverlapping(self, nums: List[int], target: int) -> int`:

In [18]:
class Solution:
    def maxNonOverlapping(self, nums: List[int], target: int) -> int:
        ans = 0
        cur = 0              # tổng kể từ điểm cắt gần nhất
        seen = {0}           # các tổng tiền tố kể từ điểm cắt (0 = tiền tố rỗng)
        for x in nums:
            cur += x
            if cur - target in seen:      # có đoạn tổng target kết thúc tại đây
                ans += 1
                cur = 0
                seen = {0}                # đoạn sau phải bắt đầu sau điểm này
            else:
                seen.add(cur)
        return ans

for nums, target, want in examples + more:
    got = Solution().maxNonOverlapping(nums, target)
    print(nums, target, "->", got, "OK" if got == want else "SAI")

[1, 1, 1, 1, 1] 2 -> 2 OK
[-1, 3, 5, 1, 4, 2, -9] 6 -> 2 OK
[-2, 6, 6, 3, 5, 4, 1, 2, 8] 10 -> 3 OK
[0, 0, 0] 0 -> 3 OK


## 10. Vì sao thuật toán đúng

**Mô hình hoá.** Mỗi đoạn tổng `target` là một khoảng `[l, r]`. Bài toán = chọn nhiều khoảng nhất đôi một không giao nhau.

**Bổ đề (chọn khoảng kết thúc sớm nhất).** Gọi `I* = [l*, r*]` là khoảng có `r*` nhỏ nhất. Với một lời giải tối ưu bất kỳ, gọi `J` là khoảng có đầu mút phải nhỏ nhất trong lời giải đó. Vì `r* ≤ r_J`, thay `J` bằng `I*`: `I*` không giao các khoảng khác của lời giải (chúng đều bắt đầu sau `r_J ≥ r*`). Số khoảng giữ nguyên ⇒ tồn tại lời giải tối ưu chứa `I*`.

**Tham lam tìm đúng `I*`.** Quét `i` tăng dần, lần đầu tiên `cur − target ∈ seen` xảy ra ở `i` nhỏ nhất có một khoảng kết thúc tại `i` ⇒ đó là `r*`. Sau khi chọn, mọi khoảng còn dùng được phải có `l > r*` — tức bài toán con trên `nums[r*+1:]`, đúng là việc reset `seen = {0}`, `cur = 0`. Quy nạp trên bài toán con ⇒ tham lam tối ưu ("kết thúc sớm" như gợi ý của đề).

**Chi tiết `seen = {0}`:** giá trị `0` đại diện "tiền tố rỗng" tại điểm cắt, cho phép đoạn bắt đầu **ngay** sau điểm cắt. Với `target = 0`, ta kiểm tra `cur ∈ seen` **trước** khi thêm `cur`, nên không bao giờ tạo đoạn rỗng.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space |
|------|------|-------|
| 1. DP `O(n²)` | `O(n²)` | `O(n)` |
| 2. DP + hash map | `O(n)` trung bình | `O(n)` |
| 3. Tham lam + set reset | `O(n)` trung bình | `O(n)` worst |
| 4. Tham lam + mốc cắt | `O(n)` trung bình | `O(n)` |
| 5. Liệt kê khoảng | `O(n² log n)` | `O(n²)` |

"Trung bình" vì tra `set`/`dict` là `O(1)` kỳ vọng. Worst case không gian của cách 3 là khi không có đoạn nào (set chứa mọi tiền tố).

## 12. Bộ test

Ví dụ đề + ví dụ thường gặp, edge case (một phần tử, `target = 0`, toàn 0, số âm, không có đoạn nào), random seed cố định so với DP `O(n²)`, và `n = 10⁵`.

In [19]:
METHODS = {
    "dp_hash": dp_hash,
    "SolutionDP": lambda a, t: SolutionDP().maxNonOverlapping(a, t),
    "greedy_reset": greedy_reset,
    "SolutionBoundary": lambda a, t: SolutionBoundary().maxNonOverlapping(a, t),
    "interval_greedy": interval_greedy,
    "Solution": lambda a, t: Solution().maxNonOverlapping(a, t),
}
edge = [([5], 5, 1), ([5], 4, 0), ([0], 0, 1), ([0, 0, 0, 0], 0, 4), ([1, -1, 1, -1], 0, 2),
        ([3, -1, 2], 2, 2), ([1, 2, 2, -3, 2], 2, 3), ([2, 2], 2, 2), ([-5, 5, -5, 5], 0, 2),
        ([1, 2, 3], 7, 0), ([10000] * 5, 20000, 2)]
for nums, target, want in examples + more + edge:
    assert brute_dp(nums, target) == want, (nums, target, brute_dp(nums, target))
    for name, f in METHODS.items():
        got = f(nums, target)
        assert got == want, (name, nums, target, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [20]:
rng = random.Random(1546)
for _ in range(1500):
    n = rng.randint(1, 40)
    lim = rng.choice([1, 3, 10, 10000])
    nums = [rng.randint(-lim, lim) for _ in range(n)]
    target = rng.choice([0, rng.randint(0, 2 * lim), abs(sum(nums[: rng.randint(1, n)]))])
    ref = brute_dp(nums, target)
    for name, f in METHODS.items():
        assert f(nums, target) == ref, (name, nums, target)
print("1500 mảng ngẫu nhiên (seed 1546): mọi cách khớp DP O(n²)")

1500 mảng ngẫu nhiên (seed 1546): mọi cách khớp DP O(n²)


In [21]:
rng2 = random.Random(9)
big = [rng2.randint(-10**4, 10**4) for _ in range(10**5)]
big0 = [0] * 10**5
for name in ["dp_hash", "greedy_reset", "SolutionBoundary", "Solution"]:
    f = METHODS[name]
    t0 = time.perf_counter()
    res = [f(big, 10), f(big0, 0), f(big, 10**6)]
    print(f"{name:16s}: {res} ({time.perf_counter() - t0:.3f}s)")
    assert res[1] == 10**5

dp_hash         : [219, 100000, 4] (0.082s)
greedy_reset    : [219, 100000, 4] (0.034s)


SolutionBoundary: [219, 100000, 4] (0.054s)
Solution        : [219, 100000, 4] (0.032s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm **mọi** đoạn có tổng `target` (bài 560)

Quen tay bài *Subarray Sum Equals K* nên đếm cả các đoạn chồng lấn.

In [22]:
from collections import Counter

def wrong_count_all(nums, target):
    cnt, cur, ans = Counter({0: 1}), 0, 0
    for x in nums:
        cur += x
        ans += cnt[cur - target]        # SAI: đếm cả đoạn chồng lấn
        cnt[cur] += 1
    return ans

for nums, target, want in examples + more:
    print(nums, target, "| sai:", wrong_count_all(nums, target), "| đúng:", want)

[1, 1, 1, 1, 1] 2 | sai: 4 | đúng: 2
[-1, 3, 5, 1, 4, 2, -9] 6 | sai: 3 | đúng: 2
[-2, 6, 6, 3, 5, 4, 1, 2, 8] 10 | sai: 3 | đúng: 3
[0, 0, 0] 0 | sai: 6 | đúng: 3


Sửa: khi tìm thấy một đoạn thì **cắt** và reset (tham lam), không cộng dồn.

### Lỗi 2 — Tìm thấy đoạn nhưng không reset `seen`

Chỉ tăng `ans` mà vẫn giữ các tiền tố cũ ⇒ đoạn sau có thể bắt đầu **trước** điểm cắt (chồng lấn).

In [23]:
def wrong_no_reset(nums, target):
    ans, cur, seen = 0, 0, {0}
    for x in nums:
        cur += x
        if cur - target in seen:
            ans += 1                    # SAI: thiếu reset seen/cur
        seen.add(cur)
    return ans

for nums, target, want in examples + more:
    print(nums, target, "| sai:", wrong_no_reset(nums, target), "| đúng:", want)

[1, 1, 1, 1, 1] 2 | sai: 4 | đúng: 2
[-1, 3, 5, 1, 4, 2, -9] 6 | sai: 3 | đúng: 2
[-2, 6, 6, 3, 5, 4, 1, 2, 8] 10 | sai: 3 | đúng: 3
[0, 0, 0] 0 | sai: 3 | đúng: 3


Sửa: `cur, seen = 0, {0}` ngay sau khi tìm thấy.

### Lỗi 3 — Quên phần tử `0` trong `seen` (ban đầu hoặc sau reset)

Thiếu `0` thì không phát hiện được đoạn bắt đầu ngay ở đầu mảng hoặc ngay sau điểm cắt.

In [24]:
def wrong_no_zero(nums, target):
    ans, cur, seen = 0, 0, set()        # SAI: phải là {0}
    for x in nums:
        cur += x
        if cur - target in seen:
            ans += 1
            cur, seen = 0, set()        # SAI: phải là {0}
        else:
            seen.add(cur)
    return ans

for nums, target in [([2, 2], 2), ([1, 1, 1, 1, 1], 2), ([5], 5)]:
    print(nums, target, "| sai:", wrong_no_zero(nums, target), "| đúng:", Solution().maxNonOverlapping(nums, target))

[2, 2] 2 | sai: 1 | đúng: 2
[1, 1, 1, 1, 1] 2 | sai: 1 | đúng: 2
[5] 5 | sai: 0 | đúng: 1


Sửa: `seen = {0}` ở cả hai chỗ.

### Lỗi 4 — Dùng cửa sổ trượt (chỉ đúng khi mọi số dương)

Thu hẹp cửa sổ khi tổng vượt `target` là sai khi có số âm: phần tử âm phía sau có thể kéo tổng về lại `target`.

In [25]:
def wrong_sliding(nums, target):
    ans, l, s = 0, 0, 0
    for r, x in enumerate(nums):
        s += x
        while s > target and l <= r:    # SAI khi có số âm
            s -= nums[l]
            l += 1
        if s == target and l <= r:
            ans += 1
            s, l = 0, r + 1
    return ans

for nums, target in [([3, -1, 2], 2), ([-1, 3, 5, 1, 4, 2, -9], 6), ([1, 1, 1, 1, 1], 2)]:
    print(nums, target, "| sai:", wrong_sliding(nums, target), "| đúng:", Solution().maxNonOverlapping(nums, target))

[3, -1, 2] 2 | sai: 0 | đúng: 2
[-1, 3, 5, 1, 4, 2, -9] 6 | sai: 2 | đúng: 2
[1, 1, 1, 1, 1] 2 | sai: 2 | đúng: 2


Sửa: dùng tổng tiền tố + hash (không phụ thuộc dấu).

### Lỗi 5 — Tham lam theo **điểm bắt đầu sớm nhất** thay vì **điểm kết thúc sớm nhất**

Lấy đoạn bắt đầu sớm nhất (rồi nhảy qua nó) có thể chiếm một khoảng dài che mất nhiều đoạn ngắn.

In [26]:
def wrong_earliest_start(nums, target):
    n, ans, i = len(nums), 0, 0
    while i < n:
        s, found = 0, -1
        for j in range(i, n):           # đoạn ngắn nhất bắt đầu tại i
            s += nums[j]
            if s == target:
                found = j
                break
        if found >= 0:
            ans += 1
            i = found + 1               # SAI: cam kết với điểm bắt đầu sớm nhất
        else:
            i += 1
    return ans

for nums, target in [([1, 2, 2, -3, 2], 2), ([-1, 3, 5, 1, 4, 2, -9], 6)]:
    print(nums, target, "| sai:", wrong_earliest_start(nums, target), "| đúng:", Solution().maxNonOverlapping(nums, target))

[1, 2, 2, -3, 2] 2 | sai: 2 | đúng: 3
[-1, 3, 5, 1, 4, 2, -9] 6 | sai: 1 | đúng: 2


Sửa: chọn đoạn có **điểm kết thúc** sớm nhất (chính là đoạn đầu tiên phát hiện được khi quét theo điểm kết thúc).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đoạn con có tổng = K"** ⇒ tổng tiền tố + hash map (`pre[i] − K` đã xuất hiện chưa).
- **"Nhiều nhất các đoạn/khoảng không chồng lấn"** ⇒ interval scheduling: tham lam theo **đầu mút phải**; khi quét trái → phải, "đầu mút phải sớm nhất" = "phát hiện sớm nhất" ⇒ cắt ngay và reset.
- Có số âm ⇒ quên cửa sổ trượt.
- DP `dp[i] = max(dp[i−1], dp[j] + 1)` với `dp` không giảm ⇒ chỉ cần `j` lớn nhất ⇒ hash map "giá trị → chỉ số gần nhất".
- Bài cùng họ: *Subarray Sum Equals K* (560), *Non-overlapping Intervals* (435), *Maximum Size Subarray Sum Equals k* (325), *Make Sum Divisible by P* (1590), *Find Two Non-overlapping Sub-arrays Each With Target Sum* (1477).

## 15. Biến thể của bài

1. **Tổng chia hết cho `k`** (thay vì bằng `target`): dùng tập **số dư** của tổng tiền tố — tham lam y hệt (bài tập 5).
2. **Trả về chính các đoạn**: ghi lại chỉ số bắt đầu bằng `dict` (bài tập 4).
3. **Mỗi đoạn có trọng số khác nhau, tối đa tổng trọng số**: tham lam không còn đúng ⇒ DP `dp[i] = max(dp[i−1], dp[j] + w)` (weighted interval scheduling).
4. **Đúng `m` đoạn với tổng độ dài nhỏ nhất**: DP 2 chiều (xem 1477 cho `m = 2`).
5. **Mảng vòng**: nhân đôi mảng nhưng giới hạn độ dài đoạn ≤ `n` và xử lý trùng lặp cẩn thận.

In [27]:
def variant_weighted(nums, target, weight):
    """Biến thể 3: mỗi đoạn [l, r] có trọng số weight(l, r); tối đa tổng trọng số (DP O(n²))."""
    n = len(nums)
    pre = [0] + list(accumulate(nums))
    dp = [0] * (n + 1)
    for i in range(1, n + 1):
        dp[i] = dp[i - 1]
        for j in range(i):
            if pre[i] - pre[j] == target:
                dp[i] = max(dp[i], dp[j] + weight(j, i - 1))
    return dp[n]

nums_v = [-1, 3, 5, 1, 4, 2, -9]
print("Trọng số = 1 (bài gốc):", variant_weighted(nums_v, 6, lambda l, r: 1))
print("Trọng số = độ dài đoạn:", variant_weighted(nums_v, 6, lambda l, r: r - l + 1), "(đoạn dài [1..6] thắng hai đoạn ngắn)")

Trọng số = 1 (bài gốc): 2
Trọng số = độ dài đoạn: 6 (đoạn dài [1..6] thắng hai đoạn ngắn)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. DP `O(n²)` | thử mọi điểm bắt đầu | `O(n²)` | `O(n)` | kiểm chứng |
| 2. DP + hash | `j` gần nhất có `pre[j] = pre[i] − t` | `O(n)` | `O(n)` | khi cần mở rộng sang có trọng số (với cấu trúc khác) |
| 3. Tham lam + set reset | cắt ngay khi phát hiện | `O(n)` | `O(n)` | **bản nộp**, ngắn nhất |
| 4. Tham lam + mốc cắt | `last[...] ≥ boundary` | `O(n)` | `O(n)` | không muốn tạo set mới |
| 5. Liệt kê khoảng | interval scheduling thuần | `O(n² log n)` | `O(n²)` | minh hoạ chứng minh |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxNonOverlapping([0,0,0], 0)` và `maxNonOverlapping([1,-1,0], 0)` trả về gì?

**Bài 2 (tính tay):** Chạy tay tham lam trên `nums = [2, -1, 1, 2, 0, 2, -2, 2]`, `target = 2`: đoạn nào được cắt và đáp án là bao nhiêu?

**Bài 3 (điền TODO):** Hoàn thiện `ex3_greedy(nums, target)`.

**Bài 4 (tự viết lại):** Viết `ex4_segments(nums, target)` trả về danh sách các đoạn `(l, r)` mà tham lam chọn (đúng số lượng tối đa, đôi một không giao, mỗi đoạn tổng `target`).

**Bài 5 (mini challenge):** Viết `ex5_divisible(nums, k)` = số đoạn không chồng lấn tối đa có **tổng chia hết cho `k`** (`k ≥ 1`), `O(n)`.

In [28]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_greedy(nums: List[int], target: int) -> int:
    ans, cur, seen = 0, 0, {0}
    for x in nums:
        cur += x
        # TODO: nếu cur - target thuộc seen: tăng ans, reset cur và seen
        # TODO: ngược lại thêm cur vào seen
        pass
    return ans

In [29]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_segments(nums: List[int], target: int):
    # TODO
    raise NotImplementedError

def ex5_divisible(nums: List[int], k: int) -> int:
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Với `target = 0`, mỗi số 0 đứng riêng là một đoạn. (2) `[0,0,0]` → 3. (3) `[1,−1,0]`: đoạn `[1,−1]` rồi `[0]` → 2.

**Bài 2.** (1) Dùng `trace_greedy`. (2) Đoạn đầu tiên là `[2]` ngay vị trí 0. (3) Các đoạn được cắt: `[0..0] = [2]`, `[1..3] = [-1,1,2]`, `[5..5] = [2]`, `[7..7] = [2]` ⇒ đáp án 4 (tại `i = 7`, `cur = 0` và `−2` đã có trong `seen`).

**Bài 3.** (1) `if cur - target in seen`. (2) `ans += 1; cur, seen = 0, {0}`. (3) `else: seen.add(cur)`.

**Bài 4.** (1) Thay `set` bằng `dict` tiền tố → chỉ số kết thúc của tiền tố. (2) Khi cắt, đoạn là `(pos[cur − target] + 1, i)`. (3) Reset `pos = {0: i}`.

**Bài 5.** (1) Tổng đoạn chia hết cho `k` ⇔ hai tiền tố cùng số dư mod `k`. (2) `seen` chứa **số dư**; điều kiện là `cur % k ∈ seen`. (3) Tham lam y hệt, kiểm chứng bằng DP `O(n²)`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
a1 = (Solution().maxNonOverlapping([0, 0, 0], 0), Solution().maxNonOverlapping([1, -1, 0], 0))
print("Bài 1:", a1)
assert a1 == (3, 2)

# Bài 2
a2 = trace_greedy([2, -1, 1, 2, 0, 2, -2, 2], 2)
assert a2 == 4 == brute_dp([2, -1, 1, 2, 0, 2, -2, 2], 2)

Bài 1: (3, 2)
 i |   x |  cur | cur-t | hành động
 0 |   2 |    2 |     0 | CẮT: đoạn [0..0] = [2] tổng 2 | ans=1
 1 |  -1 |   -1 |    -3 | thêm -1 vào seen = [-1, 0]
 2 |   1 |    0 |    -2 | thêm 0 vào seen = [-1, 0]
 3 |   2 |    2 |     0 | CẮT: đoạn [3..3] = [2] tổng 2 | ans=2
 4 |   0 |    0 |    -2 | thêm 0 vào seen = [0]
 5 |   2 |    2 |     0 | CẮT: đoạn [5..5] = [2] tổng 2 | ans=3
 6 |  -2 |   -2 |    -4 | thêm -2 vào seen = [-2, 0]
 7 |   2 |    0 |    -2 | CẮT: đoạn [7..7] = [2] tổng 2 | ans=4
Các đoạn được chọn: [(0, 0), (3, 3), (5, 5), (7, 7)]


In [31]:
# Bài 3
def ex3_greedy_sol(nums, target):
    ans, cur, seen = 0, 0, {0}
    for x in nums:
        cur += x
        if cur - target in seen:
            ans += 1
            cur, seen = 0, {0}
        else:
            seen.add(cur)
    return ans

rng3 = random.Random(3)
for _ in range(800):
    nums = [rng3.randint(-4, 4) for _ in range(rng3.randint(1, 30))]
    t = rng3.randint(0, 6)
    assert ex3_greedy_sol(nums, t) == brute_dp(nums, t)
print("Bài 3: OK")

Bài 3: OK


In [32]:
# Bài 4
def ex4_segments_sol(nums, target):
    segs, cur, pos = [], 0, {0: -1}
    for i, x in enumerate(nums):
        cur += x
        if cur - target in pos:
            segs.append((pos[cur - target] + 1, i))
            cur, pos = 0, {0: i}
        else:
            pos[cur] = i
    return segs

rng4 = random.Random(4)
for _ in range(800):
    nums = [rng4.randint(-4, 4) for _ in range(rng4.randint(1, 30))]
    t = rng4.randint(0, 6)
    segs = ex4_segments_sol(nums, t)
    assert len(segs) == brute_dp(nums, t)
    assert all(sum(nums[l:r + 1]) == t and l <= r for l, r in segs)
    assert all(segs[i][1] < segs[i + 1][0] for i in range(len(segs) - 1))
print("Bài 4: OK, ví dụ:", ex4_segments_sol([-2, 6, 6, 3, 5, 4, 1, 2, 8], 10))

Bài 4: OK, ví dụ: [(0, 2), (4, 6), (7, 8)]


In [33]:
# Bài 5
def ex5_divisible_sol(nums, k):
    ans, cur, seen = 0, 0, {0}
    for x in nums:
        cur = (cur + x) % k
        if cur in seen:
            ans += 1
            cur, seen = 0, {0}
        else:
            seen.add(cur)
    return ans

def ex5_brute(nums, k):
    n = len(nums)
    pre = [0] + list(accumulate(nums))
    dp = [0] * (n + 1)
    for i in range(1, n + 1):
        dp[i] = dp[i - 1]
        for j in range(i):
            if (pre[i] - pre[j]) % k == 0:
                dp[i] = max(dp[i], dp[j] + 1)
    return dp[n]

rng5 = random.Random(5)
for _ in range(800):
    nums = [rng5.randint(-10, 10) for _ in range(rng5.randint(1, 25))]
    k = rng5.randint(1, 7)
    assert ex5_divisible_sol(nums, k) == ex5_brute(nums, k)
print("Bài 5: OK, ví dụ:", ex5_divisible_sol([3, 1, 4, 1, 5, 9, 2, 6], 5))

Bài 5: OK, ví dụ: 2


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def maxNonOverlapping(self, nums, target):
        ans, cur, seen = 0, 0, {0}
        for x in nums:
            cur += x
            if cur - target in seen:      # đoạn kết thúc sớm nhất -> lấy ngay
                ans += 1
                cur, seen = 0, {0}        # cắt: đoạn sau bắt đầu sau đây
            else:
                seen.add(cur)
        return ans
```

**Template "nhiều đoạn con không chồng lấn thoả điều kiện tiền tố":** quét trái → phải với tập tiền tố kể từ điểm cắt; vừa phát hiện đoạn thoả ⇒ đếm và reset (tham lam đầu mút phải sớm nhất).

**Checklist trước khi submit:**
- [ ] `seen` khởi tạo **và** reset bằng `{0}`.
- [ ] Kiểm tra `cur − target ∈ seen` **trước** khi thêm `cur` (tránh đoạn rỗng khi `target = 0`).
- [ ] Reset cả `cur` lẫn `seen` khi cắt.
- [ ] Không dùng cửa sổ trượt (có số âm).
- [ ] Thử nhanh: `([1,1,1,1,1],2) → 2`, `([-1,3,5,1,4,2,-9],6) → 2`, `([0,0,0],0) → 3`.